In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time

# --- config ---
EPISODES   = 20000
LR         = 0.5
GAMMA      = 0.99
EPSILON    = 1.0
EPS_DECAY  = 0.9997
EPS_MIN    = 0.01
MAX_STEPS  = 50  # evita loop infinito na renderização

SHOW_AT = {1, 10, 50, 100, 500} | set(range(1000, 20001, 1000))

ROWS, COLS  = 4, 12
# penhasco extra: row 1, cols 3 até 8
EXTRA_HOLES = set(range(1*COLS + 3, 1*COLS + 12))  # estados 15 a 20

action_names = {0: "UP", 1: "RIGHT", 2: "DOWN", 3: "LEFT"}

class CliffWithHoles(gym.Wrapper):
    def __init__(self, extra_holes, render_mode=None):
        super().__init__(gym.make("CliffWalking-v1", render_mode=render_mode))
        self.extra_holes = extra_holes

    def step(self, action):
        state, reward, terminated, truncated, info = self.env.step(action)
        # penaliza buraco extra igual ao cliff original
        if state in self.extra_holes:
            return state, -100, True, False, info
        return state, reward, terminated, truncated, info

env_render = CliffWithHoles(EXTRA_HOLES, render_mode="rgb_array")
env_train  = CliffWithHoles(EXTRA_HOLES)
Q          = np.zeros((env_train.observation_space.n, env_train.action_space.n))

def choose_action(state, epsilon):
    if np.random.rand() < epsilon:
        return env_train.action_space.sample()
    q = Q[state]
    return np.random.choice(np.where(q == q.max())[0])

def overlay_holes(ax, frame):
    """Retângulos vermelhos sobre os buracos extras."""
    h, w   = frame.shape[:2]
    cell_h = h / ROWS
    cell_w = w / COLS
    for state in EXTRA_HOLES:
        row, col = divmod(state, COLS)
        ax.add_patch(plt.Rectangle(
            (col * cell_w, row * cell_h), cell_w, cell_h,
            color="red", alpha=0.6, zorder=2
        ))

def show(frame, ep, step, action, rewards):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    ax1.imshow(frame)
    overlay_holes(ax1, frame)
    ax1.axis("off")
    ax1.set_title(f"Ep {ep} | Passo {step} | Ação: {action_names[action]}")
    if len(rewards) >= 100:
        ax2.plot(np.convolve(rewards, np.ones(100)/100, mode="valid"), color="steelblue")
        ax2.axhline(y=-13, color="green", linestyle="--", label="ótimo (-13)")
        ax2.legend()
    else:
        ax2.plot(rewards, color="steelblue")
    ax2.set_title("Recompensa média (janela=100)")
    ax2.set_xlabel("Episódio")
    ax2.set_ylabel("Reward")
    plt.tight_layout()
    clear_output(wait=True)
    display(fig)
    plt.close(fig)
    time.sleep(0.1)

def render_optimal_route(ep, rewards):
    state, _ = env_render.reset()
    visited   = set()
    steps     = 0
    # MAX_STEPS evita loop infinito quando Q-table ainda não aprendeu
    while state != 47 and state not in visited and steps < MAX_STEPS:
        visited.add(state)
        action                             = np.argmax(Q[state])
        state, _, terminated, truncated, _ = env_render.step(action)
        steps += 1
        show(env_render.render(), ep, steps, action, rewards)
        if terminated or truncated:
            break
    return steps

# --- treino SARSA ---
rewards = []
epsilon = EPSILON

for ep in range(1, EPISODES + 1):
    state, _ = env_train.reset()
    action   = choose_action(state, epsilon)
    done     = False
    total    = 0

    while not done:
        next_state, reward, terminated, truncated, _ = env_train.step(action)
        done        = terminated or truncated
        next_action = choose_action(next_state, epsilon)

        # SARSA
        Q[state, action] += LR * (reward + GAMMA * Q[next_state, next_action] - Q[state, action])

        state  = next_state
        action = next_action
        total += reward

    epsilon = max(EPS_MIN, epsilon * EPS_DECAY)
    rewards.append(total)

    if ep in SHOW_AT:
        steps = render_optimal_route(ep, rewards)
        print(f"Ep {ep:>6} | epsilon: {epsilon:.3f} | reward médio: {np.mean(rewards[-min(100,len(rewards)):]):.1f} | passos: {steps}")

env_train.close()
env_render.close()

# --- gráfico final ---
plt.figure(figsize=(12, 4))
plt.plot(np.convolve(rewards, np.ones(100)/100, mode="valid"), color="steelblue")
plt.axhline(y=-13, color="green", linestyle="--", label="ótimo (-13)")
plt.title("SARSA + penhasco extra — Recompensa média (janela=100)")
plt.xlabel("Episódio")
plt.ylabel("Reward")
plt.legend()
plt.tight_layout()
plt.show()

print(f"\nRecompensa média final (últimos 100): {np.mean(rewards[-100:]):.1f}")

In [3]:
category_urls = {
    'Pet Shop': 'https://www.araujo.com.br/pet-shop',
    'Higiene e Beleza': 'https://www.araujo.com.br/higiene-e-beleza',
    'Medicamentos': 'https://www.araujo.com.br/medicamentos'
}

print("Category URLs dictionary created.")
print(category_urls)

Category URLs dictionary created.
{'Pet Shop': 'https://www.araujo.com.br/pet-shop', 'Higiene e Beleza': 'https://www.araujo.com.br/higiene-e-beleza', 'Medicamentos': 'https://www.araujo.com.br/medicamentos'}


In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Initialize an empty list to store all scraped data
all_products_data = []

# Iterate through each category URL
for category_name, base_url in category_urls.items():
    print(f"\n--- Scraping category: {category_name} ---")
    current_page_url = base_url
    page_counter = 1

    while True:
        print(f"Scraping page {page_counter} from: {current_page_url}")

        try:
            response = requests.get(current_page_url)
            response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
            soup = BeautifulSoup(response.text, 'html.parser')

            products = soup.find_all('div', class_='productTile')

            if products:
                for product in products:
                    product_name = product.get('title', 'N/A')
                    price_element = product.find('span', class_='productPrice__price')
                    product_price = price_element.get_text(strip=True) if price_element else 'N/A'
                    all_products_data.append({
                        'Produto': product_name,
                        'Preço': product_price,
                        'Categoria': category_name
                    })
            else:
                print(f"No products found on page {page_counter} for category {category_name}. Stopping pagination.")
                break

            # Find the next page link
            next_page_link = None
            pagination_footer = soup.find('div', class_='productGrid__footer')
            if pagination_footer:
                next_button = pagination_footer.find('li', class_='next-page')
                if next_button:
                    next_page_link = next_button.find('a')
                if not next_page_link:
                     for link in pagination_footer.find_all('a'):
                         if 'próxima' in link.get_text(strip=True).lower(): # Assuming Portuguese for 'next'
                             next_page_link = link
                             break

            if next_page_link and next_page_link.get('href'):
                current_page_url = requests.compat.urljoin(base_url, next_page_link['href'])
                page_counter += 1
            else:
                print(f"No next page link found on page {page_counter} for category {category_name}. Finishing scraping.")
                break # No next page link, so this is the last page

        except requests.exceptions.RequestException as e:
            print(f"Error fetching URL {current_page_url} for category {category_name}: {e}")
            break
        except Exception as e:
            print(f"An error occurred on page {current_page_url} for category {category_name}: {e}")
            break

# Create DataFrame from all collected data
df_araujo_all_products = pd.DataFrame(all_products_data)

# Display head of the DataFrame and total count
display(df_araujo_all_products.head())
print(f"Total products scraped across all categories: {len(df_araujo_all_products)}")

# Save to CSV
csv_output_araujo_all = df_araujo_all_products.to_csv(index=False)
with open('araujo_all_products.csv', 'w', encoding='utf-8') as f:
    f.write(csv_output_araujo_all)
print("Data saved to araujo_all_products.csv")


--- Scraping category: Pet Shop ---
Scraping page 1 from: https://www.araujo.com.br/pet-shop
Error fetching URL https://www.araujo.com.br/pet-shop for category Pet Shop: 403 Client Error: Forbidden for url: https://www.araujo.com.br/pet-shop

--- Scraping category: Higiene e Beleza ---
Scraping page 1 from: https://www.araujo.com.br/higiene-e-beleza
Error fetching URL https://www.araujo.com.br/higiene-e-beleza for category Higiene e Beleza: 403 Client Error: Forbidden for url: https://www.araujo.com.br/higiene-e-beleza

--- Scraping category: Medicamentos ---
Scraping page 1 from: https://www.araujo.com.br/medicamentos
Error fetching URL https://www.araujo.com.br/medicamentos for category Medicamentos: 403 Client Error: Forbidden for url: https://www.araujo.com.br/medicamentos


""


Total products scraped across all categories: 0
Data saved to araujo_all_products.csv
